# 01 · Data ingestion: Amazon Product Dataset 2020 → `products.parquet`

**Checkpoint 1 deliverable.** This notebook loads the raw Kaggle CSV (PromptCloud, 10,002 products), profiles it, picks the **Toys & Games** slice, normalizes it into the project schema and writes:

| File | Contents |
|---|---|
| `data/processed/products.parquet` | the Toys & Games slice indexed for RAG |
| `data/processed/products_all.parquet` | every product, normalized |
| `data/processed/ingest_report.json` | row counts, field coverage, warnings |

All logic lives in `src/voiceshop/data/ingest.py`, so `python scripts/build_data.py` reproduces this notebook exactly.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from voiceshop.data import ingest

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)
RAW_DIR, OUT_DIR = ROOT / "data" / "raw", ROOT / "data" / "processed"
csvs = sorted(RAW_DIR.glob("*.csv"))
assert csvs, f"Put the Kaggle CSV in {RAW_DIR}"
RAW_CSV = csvs[0]
RAW_CSV.name

'marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv'

## 1 · Load and profile
The raw file has 28 columns. The profile shows which fields we can rely on: there are no ratings, brands, ingredients or descriptions.

In [2]:
raw = ingest.load_raw(RAW_CSV)
print(raw.shape)
prof = ingest.profile(raw)
prof

(10002, 28)


,column,filled,fill_rate,example
0,uniq_id,10002,1.000,4c69b61db1fc16e7013b43fc926e502d
1,product_name,10002,1.000,"DB Longboards CoreFlex Crossbow 41"" Bamboo Fiberglass Longboard Complete"
2,product_url,10002,1.000,https://www.amazon.com/DB-Longboards-CoreFlex-Fiberglass-Longboard/dp/B07KMV...
3,image,10002,1.000,https://images-na.ssl-images-amazon.com/images/I/51j3fPQTQkL.jpg|https://ima...
4,is_amazon_seller,10002,1.000,Y
5,selling_price,9895,0.989,$237.68
6,about_product,9729,0.973,Make sure this fits by entering your model number. | RESPONSIVE FLEX: The Cr...
7,technical_details,9212,0.921,The snap circuits mini kits classpack provides basic electric circuitry info...
8,category,9172,0.917,"Sports & Outdoors | Outdoor Recreation | Skates, Skateboards & Scooters | Sk..."
9,shipping_weight,8864,0.886,10.7 pounds


In [3]:
empty = prof[prof.fill_rate == 0].column.tolist()
print(f"{len(empty)} of {raw.shape[1]} columns are completely empty:", empty)

13 of 28 columns are completely empty: ['asin', 'quantity', 'list_price', 'brand_name', 'sku', 'stock', 'product_details', 'color', 'dimensions', 'ingredients', 'direction_to_use', 'size_quantity_variant', 'product_description']


## 2 · Choosing the slice
The brief suggests Household Cleaning, but this sample is mostly toys. Health & Household has only 23 products. A cleaning filter matches about 10 items, and none of them are cleaners: laundry hampers, drinking straws, and a toy set of "Polish" soldiers. **Toys & Games has 6,662 products**, so it's the slice with enough depth for retrieval and comparison.

In [4]:
ingest.top_categories(raw, depth=1, n=15)

category
Toys & Games                                6662
(none)                                       830
Home & Kitchen                               708
Clothing, Shoes & Jewelry                    630
Sports & Outdoors                            540
Baby Products                                214
Arts, Crafts & Sewing                        124
Office Products                               77
Hobbies                                       34
Industrial & Scientific                       29
Health & Household                            23
Remote & App Controlled Vehicle Parts         22
Tools & Home Improvement                      17
Remote & App Controlled Vehicles & Parts      17
Pet Supplies                                  16
Name: count, dtype: int64

In [5]:
cleaning = ingest.select_slice(ingest.normalize(raw, ingest.SliceConfig.household_cleaning()),
                               ingest.SliceConfig.household_cleaning())
print(len(cleaning), "products match the Household Cleaning filter")
cleaning.title.head(10)

10 products match the Household Cleaning filter


0    Sweet Jojo Designs Vintage Floral Boho Baby Kid Clothes Laundry Hamper - Blu...
1    Aardvark 61500210 Unwrapped Paper Giant Drinking Straw, 9/32" Diameter x 7-3...
2    Sweet Jojo Designs Blush Pink, Mint Green and White Boho Baby Kid Clothes La...
3                        Nickelodeon Paw Patrol Pop Up Hamper, 13.5" x 13.5" x 21.5"
4    Sweet Jojo Designs Baby Children Kids Clothes Laundry Hamper for Grey and Wh...
5                                 LAB SERIES Max Ls Daily Renewing Cleanser, 5 Ounce
6                                         COBI Historical Collection Polish Soldiers
7    Sweet Jojo Designs Green and Beige Rustic Deer Buffalo Plaid Check Baby Kid ...
8    Sweet Jojo Designs Baby Children Kids Clothes Laundry Hamper for Turquoise a...
9      American Educational Vinyl Clever Catch Food and Nutrition Ball, 24" Diameter
Name: title, dtype: str

In [6]:
toys_raw = raw[raw.category.fillna("").str.startswith("Toys & Games")]
toys_raw.category.map(lambda s: ingest.category_parts(s)[1] if len(ingest.category_parts(s)) > 1 else "(top)").value_counts().head(15)

category
Hobbies                         905
Games & Accessories             645
Arts & Crafts                   539
Stuffed Animals & Plush Toys    495
Party Supplies                  487
Dress Up & Pretend Play         451
Sports & Outdoor Play           417
Learning & Education            377
Puzzles                         364
Toy Figures & Playsets          358
Dolls & Accessories             313
Baby & Toddler Toys             257
Novelty & Gag Toys              230
Play Vehicles                   210
Building Toys                   207
Name: count, dtype: int64

## 3 · Normalize to the target schema

| Field | How it's derived |
|---|---|
| `sku` | ASIN parsed from the squashed spec text (`...|ASIN:B07Q1JVZMW|...`) |
| `age_min_years` / `age_max_years` | "Manufacturer recommended age" from the spec, else "Ages 8+" from the title or features (`age_source`) |
| `price` / `price_max` | Selling price; a range like "$9.99 - $24.99" is split into min and max |
| `brand` | Longest title prefix shared by many titles (`brand_source = title_guess`) |
| `piece_count`, `players_min` / `players_max` | Parsed from the title or features |
| `materials`, `attributes`, `hazards` | Tags mined from text using the toys domain pack. Hazards feed the safety guardrail. |
| `doc_id` | Stable citation key (`amz2020-…`) |

In [7]:
cfg = ingest.SliceConfig.toys_and_games()
allp = ingest.normalize(raw, cfg)
toys = ingest.select_slice(allp, cfg)
print(len(allp), "products after cleaning and de-duplication;", len(toys), "in the toys slice")
toys[["doc_id", "sku", "title", "brand", "price", "age_min_years", "age_max_years", "piece_count", "attributes"]].sample(10, random_state=0)

9978 products after cleaning and de-duplication; 6658 in the toys slice


,doc_id,sku,title,brand,price,age_min_years,age_max_years,piece_count,attributes
4869,amz2020-7d6a3ff349,B071L7S6ZF,BANZAI Spin Out Slide with Spin Disk Bodyboard,BANZAI,14.12,5.0,NaN,NaN,[]
944,amz2020-ee11b8d479,B06XH52JW3,ALEX Toys Future Coders Robot Races Coding Skills Kit,Alex,15.66,4.0,8.0,NaN,[stem_educational]
2462,amz2020-6b22fc7466,B00VNW9BUE,Redcat Racing Servo Mount,Redcat Racing,7.88,14.0,NaN,NaN,[]
3791,amz2020-a56afaf6e5,B075YP6SHX,Costume Sunglasses Winnie The Pooh Sun-Staches Party Favors UV400,Costume Sunglasses,8.91,1.5,NaN,NaN,[pretend_play]
6633,amz2020-0c35212746,B00L3WWWCS,"LeapFrog AlphaPup Toy (Frustration Free Packaging), Green",LeapFrog,13.99,12.0,NaN,NaN,[]
3355,amz2020-29548bbe63,B01897IQOM,"Charizard 2"" Album with 100 Ultra Pro Platinum 9-Pocket Sheets",Charizard,29.99,8.0,15.0,NaN,[]
6614,amz2020-d6c20cba2c,B07QYXCJH4,Lil' Rider 2-in-1 Convertible Scooter for Toddlers & Children – Adjustable S...,Lil',47.80,3.0,6.0,NaN,"[outdoor, stem_educational]"
3354,amz2020-ac3c8f0420,B076WVMBGY,"Wild Republic Audubon Birds Collection with Authentic Bird Sounds, Hummingbi...",Wild Republic,24.99,NaN,NaN,NaN,"[plush, stem_educational]"
2542,amz2020-ff135b9c41,B0798GNV9L,Educational Insights Geosafari Jr. Butterfly Bungalow: Life Science Introduc...,Educational Insights,10.92,4.0,7.0,NaN,[stem_educational]
1949,amz2020-4411db2ab6,B07MCVVDX1,Shrinky Dinks Creative Pack 25 Sheets Crystal Clear Kids Art and Craft Activity,Shrinky Dinks,16.19,5.0,NaN,NaN,[arts_crafts]


## 4 · Coverage of the fields the agent will filter and compare on

In [8]:
pd.Series(ingest.coverage(toys), name="toys coverage").to_frame()

,toys coverage
price,0.988
sku,0.873
age_min_years,0.742
piece_count,0.115
players,0.035
materials,0.322
has_hazard_tag,0.049
size_oz,0.053
price_per_oz,0.053
rating,0.000


In [9]:
print("Price ($):"); print(toys.price.describe().round(2).to_string())
print(f"\nUnder $30: {(toys.price < 30).mean():.0%}   Over $300: {(toys.price > 300).sum()} products   No price: {toys.price.isna().sum()}")

Price ($):
count    6577.00
mean       34.26
std       130.85
min         0.01
25%         9.99
50%        15.61
75%        26.40
max      5332.00

Under $30: 78%   Over $300: 84 products   No price: 81


In [10]:
bins = [-0.1, 2.9, 5.9, 8.9, 11.9, 13.9, 100]
labels = ["0-2", "3-5", "6-8", "9-11", "12-13", "14+"]
pd.cut(toys.age_min_years, bins=bins, labels=labels).value_counts(dropna=False).sort_index().rename("min age (years)").to_frame()

,min age (years)
age_min_years,
0-2,888
3-5,1993
6-8,882
9-11,163
12-13,302
14+,712
NaN,1718


In [11]:
print("Attribute tags:"); print(toys.attributes.explode().value_counts().to_string())
print("\nHazard tags (feed the safety guardrail):"); print(toys.hazards.explode().value_counts().to_string())

Attribute tags:
attributes
pretend_play          614
stem_educational      556
plush                 501
outdoor               453
puzzle                417
arts_crafts           400
wooden                379
collectible           320
building              304
non_toxic             258
board_or_card_game    257
washable              223
eco_friendly          123
remote_control        116
batteries_required     86
batteries_included     49
no_batteries           25

Hazard tags (feed the safety guardrail):
hazards
magnets              133
balloons              88
projectile            51
choking_hazard        40
not_for_under_3       12
adult_supervision     11
button_battery         3
heat_source            1


In [12]:
toys.brand.value_counts().head(20).rename("products (brand guessed from title)").to_frame()

,products (brand guessed from title)
brand,
MightySkins,182
Amscan,156
Melissa & Doug,125
Disney,90
Bachmann,71
Rubie's,71
LEGO,55
Trends International,54
Funko Pop!,52


## 5 · What gets embedded
`embed_text` combines the title, brand, category path, features, age, pieces, players, materials and tags. Phase 2 embeds this text.

In [13]:
print(toys.embed_text.str.len().describe().round(0).to_string())
print()
print(toys.embed_text.iloc[42])

count    6658.0
mean      573.0
std       327.0
min        90.0
25%       342.0
50%       473.0
75%       713.0
max      1500.0

Domez My Hero Academia 4-Piece Bundle
Brand: Domez
Category: Toys & Games > Toy Figures & Playsets > Action Figures
Features: 3.1” Domez feature character-authentic details and dynamic styling that collectors crave; Each mystery 4-Pack includes 4 collectible characters, Each enclosed in their own stackable & connectable display case; Nine styles available, including all might, izuku midoriya (deku), Katsuki Baku go, ochaco uraraka, Tenya iida, Shoto todoroki, tsuyu asui, shot a aizwa (eraser head) and a mystery Chase variant!; Suitable for ages 8 and up
Recommended age: 8+ years
Pieces: 4
Attributes: collectible


## 6 · Write outputs

In [14]:
rep = ingest.run(RAW_CSV, OUT_DIR, cfg)
print(rep.to_json())
for w in rep.warnings:
    print("WARNING:", w)

{
  "raw_rows": 10002,
  "normalized_rows": 9978,
  "slice_rows": 6658,
  "slice_name": "toys_and_games",
  "coverage": {
    "price": 0.988,
    "sku": 0.873,
    "age_min_years": 0.742,
    "piece_count": 0.115,
    "players": 0.035,
    "materials": 0.322,
    "has_hazard_tag": 0.049,
    "size_oz": 0.053,
    "price_per_oz": 0.053,
    "rating": 0.0,
    "brand_guessed_from_title": 1.0,
    "has_attributes": 0.558,
    "has_features": 0.974,
    "url": 1.0
  },
  "warnings": [
    "No ratings in the dataset: rank by price/attributes, or fetch ratings via web.search."
  ]
}


## 7 · Decisions and limitations

- **Slice:** Toys & Games (6,658 products after de-duplication). Household Cleaning was dropped because this sample has essentially no cleaning products.
- **No ratings.** Rank by constraint match, price and age fit, or fetch live ratings with `web.search` and label them as live.
- **No brand field.** Brand is guessed from the title, so brand filters must be fuzzy.
- **No ingredients or descriptions.** `materials` and the tags are mined from text, and the answerer must not claim more than they say.
- **Recommended age** is available for about 74% of toys. Unknown age is treated as unknown, never as safe.
- **Price outliers:** about 80 items cost over $300 (playsets, lab equipment). They stay in the catalog; budget filters handle them.
- **Reviews:** the dataset has none, so `reviews.parquet` is skipped (it's optional in the brief).